In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
from tqdm import tqdm
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

data_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
print(df.shape) #samples and featuers count
df.describe()

In [ ]:
# Task 1: Write your code here(Handle missing values appropriately):
# Missing values
print("Missing values:")
print(df.isnull().sum())

all_cols = df.columns
print(list(all_cols))
df = df.dropna(subset=[all_cols])

print("-"*40)

print("Missing values:")
print(df.isnull().sum())

In [ ]:
# 2. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Do we have categorical columns?")
print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Target")

scaler = StandardScaler()

# scale
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Task 5: Write your code here:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['Target'], bins=50, edgecolor='black', color='skyblue')
axes[0].set_title('Target Distribution (look imbalanced)')
axes[0].set_xlabel('Target')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['P_2'], bins=50, edgecolor='black', color='salmon')
axes[1].set_title('P_2 Distribution')
axes[1].set_xlabel('P_2')
axes[1].set_ylabel('Frequency')

axes[2].hist(df['D_87'], bins=30, edgecolor='black', color='lightgreen')
axes[2].set_title('D_87 Distribution (almost all NAN)')
axes[2].set_xlabel('D_87')
axes[2].set_ylabel('Frequency')

plt.tight_layout() # This stops the titles from overlapping
plt.show()

print("Rule of thumb: If class distributions are not equal, then our data is imbalanced. ")

In [ ]:
# Task 1: Write your code here:
# X, y initilize

X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float) # play around with this float to get different accuracym, might be higher

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostRegressor

models = {
  "CatBoost": CatBoostRegressor(verbose=0)
}



# cl train

all_results = {}

for name in models:
  all_results[name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}


n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    #------ in multi class we use theesee:
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['precision'].append(precision)
    all_results[model_name]['recall'].append(recall)
    all_results[model_name]['f1'].append(f1)


    accuracy = accuracy_score(y_test, y_pred)
    precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)


# show results
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  Accuracy:  {np.mean(all_results[model_name]['accuracy']):.4f}")
  print(f"  Precision: {np.mean(all_results[model_name]['precision']):.4f}")
  print(f"  Recall:    {np.mean(all_results[model_name]['recall']):.4f}")
  print(f"  F1-Score:  {np.mean(all_results[model_name]['f1']):.4f}")



In [ ]:
feature_cols = df.drop("Target", axis=1).columns
print(list(feature_cols))

In [ ]:

# Task 1: Write your code here:
# Feature importance
feature_cols = df.drop("Target", axis=1).columns
print(list(feature_cols))

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(22, 12))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print("P_2 is the golden feature")

In [ ]:
# Task Bonus: Write your code here: